In [ ]:
import carla #the sim library itself
import random #to pick random spawn point
import cv2 #to work with images from cameras
import numpy as np #in this example to change image representation - re-shaping
import csv
import time
import os
import logging
from queue import Queue
from queue import Empty
from matplotlib import cm # Diperlukan untuk colormap proyeksi
from PIL import Image # Diperlukan untuk menyimpan gambar proyeksi

In [ ]:
# # connect to the sim
# client = carla.Client('localhost', 2000)

try:
    # Connect to the server on localhost at port 2000
    client = carla.Client('localhost', 2000)
    client.set_timeout(10.0)  # seconds

    # Retrieve the worl
    print("Connected to CARLA world:")

except Exception as e:
    print("Failed to connect to CARLA:", e)

Connected to CARLA world:


In [ ]:
#ganti town
client.load_world('Town04')
world = client.get_world()

#ganti cuaca
weather = carla.WeatherParameters.CloudyNight
world.set_weather(weather)
spawn_points = world.get_map().get_spawn_points()
print(f"Found {len(spawn_points)} spawn points.")

Found 372 spawn points.


In [ ]:
CITYSCAPES_LEGEND = {
    (0, 0, 0): 0, (70, 70, 70): 1, (100, 40, 40): 2, (55, 90, 80): 3,
    (220, 20, 60): 4, (153, 153, 153): 5, (157, 234, 50): 6, (128, 64, 128): 7,
    (244, 35, 232): 8, (107, 142, 35): 9, (0, 0, 142): 10, (102, 102, 156): 11,
    (220, 220, 0): 12, (70, 130, 180): 13, (81, 0, 81): 14, (150, 100, 100): 15,
    (230, 150, 140): 16, (180, 165, 180): 17, (250, 170, 30): 18,
    (110, 190, 160): 19, (170, 120, 50): 20, (45, 60, 150): 21, (145, 170, 100): 22
}

In [ ]:
# Cell 1: Setup Traffic Manager
tm_port = 8000
traffic_manager = client.get_trafficmanager(tm_port)
traffic_manager.set_global_distance_to_leading_vehicle(2.5)

# Optional: enable hybrid mode for large maps
traffic_manager.set_hybrid_physics_mode(True)
traffic_manager.set_hybrid_physics_radius(70.0)
print("Traffic Manager configured.")

Traffic Manager configured.


In [ ]:
# Cell 2: Configure synchronous mode
settings = world.get_settings()
synchronous_master = False

if not settings.synchronous_mode:
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.05
    synchronous_master = True
    world.apply_settings(settings)
print(f"Synchronous mode enabled: {settings.synchronous_mode}, fixed delta seconds: {settings.fixed_delta_seconds}")

Synchronous mode enabled: True, fixed delta seconds: 0.05


In [ ]:
# Cell 3: Prepare spawn points and vehicle blueprints
import random

vehicle_blueprints = world.get_blueprint_library().filter('vehicle.*')

random.shuffle(spawn_points)

number_of_vehicles = min(30, len(spawn_points))
print(f"Number of vehicles to spawn: {number_of_vehicles}")


Number of vehicles to spawn: 30


In [ ]:
# Cell 4: Spawn vehicles with autopilot
vehicles_list = []
batch = []

for n in range(number_of_vehicles):
    bp = random.choice(vehicle_blueprints)
    if bp.has_attribute('color'):
        color = random.choice(bp.get_attribute('color').recommended_values)
        bp.set_attribute('color', color)
    if bp.has_attribute('driver_id'):
        driver_id = random.choice(bp.get_attribute('driver_id').recommended_values)
        bp.set_attribute('driver_id', driver_id)
    bp.set_attribute('role_name', 'autopilot')
    transform = spawn_points[n]
    if n == 0:
        # For the first vehicle: only spawn (no autopilot yet)
        batch.append(carla.command.SpawnActor(bp, transform))
    else:
        # For others: spawn and set autopilot immediately
        batch.append(carla.command.SpawnActor(bp, transform).then(
            carla.command.SetAutopilot(carla.command.FutureActor, True, tm_port)
        ))

responses = client.apply_batch_sync(batch, synchronous_master)
for response in responses:
    if response.error:
        import logging
        logging.error(response.error)
    else:
        vehicles_list.append(response.actor_id)

print(f"Spawned {len(vehicles_list)} vehicles.")


Spawned 30 vehicles.


In [ ]:
# Cell 5: Prepare to spawn walkers
walker_blueprints = world.get_blueprint_library().filter('walker.pedestrian.*')
number_of_walkers = 10
walkers_list = []
spawn_points = []

for _ in range(number_of_walkers):
    loc = world.get_random_location_from_navigation()
    if loc is not None:
        spawn_points.append(carla.Transform(loc))

print(f"Walker spawn points generated: {len(spawn_points)}")

Walker spawn points generated: 10


In [ ]:
# Cell 6: Spawn walkers (pedestrians)
walker_speed = []
batch = []

for spawn_point in spawn_points:
    walker_bp = random.choice(walker_blueprints)
    if walker_bp.has_attribute('is_invincible'):
        walker_bp.set_attribute('is_invincible', 'false')
    speed = walker_bp.get_attribute('speed').recommended_values[1]
    walker_speed.append(float(speed))
    batch.append(carla.command.SpawnActor(walker_bp, spawn_point))

results = client.apply_batch_sync(batch, True)
for i, result in enumerate(results):
    if result.error:
        logging.error(result.error)
    else:
        walkers_list.append({"id": result.actor_id})
print(f"Spawned {len(walkers_list)} walkers.")


ERROR:root:Spawn failed because of collision at spawn position
ERROR:root:Spawn failed because of collision at spawn position


Spawned 8 walkers.


In [ ]:
# Cell 7: Spawn controllers for walkers
controller_bp = world.get_blueprint_library().find('controller.ai.walker')
batch = []

for walker in walkers_list:
    batch.append(carla.command.SpawnActor(controller_bp, carla.Transform(), walker["id"]))

results = client.apply_batch_sync(batch, True)
for i, result in enumerate(results):
    if result.error:
        logging.error(result.error)
    else:
        walkers_list[i]["con"] = result.actor_id

print("Spawned controllers for walkers.")


Spawned controllers for walkers.


In [ ]:
# Cell 8: Start walkers and assign random destinations
all_id = []
for w in walkers_list:
    all_id.extend([w["con"], w["id"]])
all_actors = world.get_actors(all_id)

world.tick()

for i in range(0, len(all_id), 2):
    controller = all_actors[i]
    actor = all_actors[i + 1]
    controller.start()
    controller.go_to_location(world.get_random_location_from_navigation())
    controller.set_max_speed(walker_speed[i // 2])

print(f"Started {len(walkers_list)} walkers with destinations.")


Started 8 walkers with destinations.


In [ ]:
# Cell 2.1: Keep ticking the world in synchronous mode
import time

print("Starting ticking loop. Press Interrupt (Ctrl+C) to stop.")
try:
    while True:
        world.tick()
        time.sleep(0.05)  # match fixed_delta_seconds to avoid busy wait
except KeyboardInterrupt:
    print("Ticking loop stopped by user.")

Starting ticking loop. Press Interrupt (Ctrl+C) to stop.
Ticking loop stopped by user.


In [ ]:
# move simulator view to the car
spawn_points = world.get_map().get_spawn_points()

#ganti spawn point mobil
start_point = spawn_points[37]
spectator = world.get_spectator()
start_point.location.z = start_point.location.z+1 #start_point was used to spawn the car but we move 1m up to avoid being on the floor
spectator.set_transform(start_point)

In [ ]:
#look for a blueprint of Mini car
vehicle_bp = world.get_blueprint_library().filter('*Audi*')

In [ ]:
#spawn a car in a random location (first spawn point in the list)
vehicle = world.try_spawn_actor(vehicle_bp[0], start_point)


In [ ]:
# Setup bird eye view

CAMERA_POS_Z = 15       # 15 meters above ground
CAMERA_ROT_X = -90.0    # Camera looks straight down
image_size_x_input = '640'
image_size_y_input = '360'


# === Get blueprint library ===
blueprint_library = world.get_blueprint_library()

# === RGB Camera blueprint ===
camera_rgb_bp = blueprint_library.find('sensor.camera.rgb')
camera_rgb_bp.set_attribute('image_size_x', image_size_x_input)
camera_rgb_bp.set_attribute('image_size_y', image_size_y_input)

# === Camera Transform ===
camera_init_trans = carla.Transform(
    carla.Location(z=CAMERA_POS_Z),
    carla.Rotation(pitch=CAMERA_ROT_X)
)

# === Spawn RGB Camera actor ===
camera_rgb = world.spawn_actor(camera_rgb_bp, camera_init_trans, attach_to=vehicle)

# === Semantic Segmentation Camera blueprint ===
camera_seg_bp = blueprint_library.find('sensor.camera.semantic_segmentation')
camera_seg_bp.set_attribute('image_size_x', image_size_x_input)
camera_seg_bp.set_attribute('image_size_y', image_size_y_input)

# === Spawn Semantic Segmentation Camera actor ===
camera_seg = world.spawn_actor(camera_seg_bp, camera_init_trans, attach_to=vehicle)

# === Lidar Camera blueprint ===
camera_lidar_bp = blueprint_library.find('sensor.lidar.ray_cast')
# camera_lidar_bp = world.get_blueprint_library().find('sensor.lidar.ray_cast_semantic') # BENAR
camera_lidar_bp.set_attribute('channels', '32')
camera_lidar_bp.set_attribute('range', '50')
camera_lidar_bp.set_attribute('rotation_frequency', '10')

# === Spawn Lidar Camera actor ===
camera_lidar = world.spawn_actor(camera_lidar_bp, camera_init_trans, attach_to=vehicle)

# 2. Spawn LiDAR SEMANTIK (Khusus Log CSV) - TAMBAHAN
lidar_sem_bp = blueprint_library.find('sensor.lidar.ray_cast_semantic')
lidar_sem_bp.set_attribute('channels', '32')
lidar_sem_bp.set_attribute('range', '50')
camera_lidar_sem = world.spawn_actor(lidar_sem_bp, camera_init_trans, attach_to=vehicle)

# === Depth Camera blueprint ===
camera_depth_bp = blueprint_library.find('sensor.camera.depth')
camera_depth_bp.set_attribute('image_size_x', image_size_x_input)
camera_depth_bp.set_attribute('image_size_y', image_size_y_input)

# === Spawn Depth Camera actor ===
camera_depth = world.spawn_actor(camera_depth_bp, camera_init_trans, attach_to=vehicle)


In [ ]:
# Setup window car view

CAMERA_POS_Z = 1.5       # 15 meters above ground
CAMERA_POS_X = 2

# === Camera Transform ===
camera_init_trans_window = carla.Transform(
    carla.Location(x=CAMERA_POS_X, z=CAMERA_POS_Z)
)

# === Spawn RGB Camera actor ===
camera_rgb_window = world.spawn_actor(camera_rgb_bp, camera_init_trans_window, attach_to=vehicle)

# === Spawn Semantic Segmentation Camera actor ===
camera_seg_window = world.spawn_actor(camera_seg_bp, camera_init_trans_window, attach_to=vehicle)

# === Spawn Lidar Camera actor ===
camera_lidar_window = world.spawn_actor(camera_lidar_bp, camera_init_trans_window, attach_to=vehicle)

camera_lidar_sem = world.spawn_actor(lidar_sem_bp, camera_init_trans_window, attach_to=vehicle)

# === Spawn Depth Camera actor ===
camera_depth_window = world.spawn_actor(camera_depth_bp, camera_init_trans_window, attach_to=vehicle)


In [ ]:
os.makedirs('D:/KP_Alexis_Doci/lidar_frames', exist_ok=True)

In [ ]:
import math
from datetime import datetime

# Definisi mapping semantik 0-22
object_id_map = {
    "None": 0, "Buildings": 1, "Fences": 2, "Other": 3, "Pedestrians": 4,
    "Poles": 5, "RoadLines": 6, "Roads": 7, "Sidewalks": 8, "Vegetation": 9,
    "Vehicles": 10, "Wall": 11, "TrafficsSigns": 12, "Sky": 13, "Ground": 14,
    "Bridge": 15, "RailTrack": 16, "GuardRail": 17, "TrafficLight": 18,
    "Static": 19, "Dynamic": 20, "Water": 21, "Terrain": 22
}
key_list = list(object_id_map.keys())
value_list = list(object_id_map.values())

# Path File Output (mengikuti base_path Anda)
base_path = "D:/KP_Alexis_Doci"
RAW_LIDAR_CSV = f"{base_path}/lidar_points_raw.csv"
DETECTION_LOG_CSV = f"{base_path}/object_log.csv"
INFO_LOG_TXT = f"{base_path}/simulation_info.txt"

# Inisialisasi Header CSV
with open(RAW_LIDAR_CSV, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp", "x", "y", "z", "object_tag"])

with open(DETECTION_LOG_CSV, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp"] + key_list)

# Catat Waktu Mulai untuk metadata
sim_start_time = datetime.now()

In [ ]:
# bird eye view
os.makedirs('D:/KP_Alexis_Doci/rgb', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/seg', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/rgb', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/seg', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/lidar', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/depth', exist_ok=True)

# window car view
os.makedirs('D:/KP_Alexis_Doci/rgb_window', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/seg_window', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/lidar_window', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/depth_window', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/lidar_frames', exist_ok=True)

os.makedirs('D:/KP_Alexis_Doci/depth_raw', exist_ok=True)
os.makedirs('D:/KP_Alexis_Doci/depth_window_raw', exist_ok=True)

rgb_frame_counter = 1
seg_frame_counter = 1
depth_frame_counter = 1
lidar_frame_counter = 1

rgb_frame_counter_window = 1
seg_frame_counter_window = 1
depth_frame_counter_window = 1
lidar_frame_counter_window = 1

# === Bird Eye view callbacks ===
def process_rgb(image):
    global rgb_frame_counter
    image.save_to_disk(f'D:/KP_Alexis_Doci/rgb/frame_{rgb_frame_counter:05d}.png')
    rgb_frame_counter += 1

def process_seg(image):
    global seg_frame_counter

    # 1. Konversi ke CityScapesPalette dan Simpan PNG Visual
    image.convert(carla.ColorConverter.CityScapesPalette)
    image.save_to_disk(f'D:/KP_Alexis_Doci/seg/frame_{seg_frame_counter:05d}.png')

    # 2. Ambil gambar yang baru disimpan dan Petakan ke Class ID
    # Simpan Visual ke disk, lalu baca lagi untuk mendapatkan warna yang dipetakan.
    visual_path = f'D:/KP_Alexis_Doci/seg/frame_{seg_frame_counter:05d}.png'
    visual_img = Image.open(visual_path).convert('RGB')
    visual_array = np.array(visual_img)
    mapped_array = np.zeros((image.height, image.width), dtype=np.uint8)

    # Lakukan Mapping
    for rgb_color, class_id in CITYSCAPES_LEGEND.items():
        mask = np.all(visual_array == rgb_color, axis=-1)
        mapped_array[mask] = class_id

    # 3. Simpan Class ID Array ke NPY
    np.save(f'D:/KP_Alexis_Doci/seg_raw/frame_{seg_frame_counter:05d}.npy', mapped_array)

    seg_frame_counter += 1

def process_depth(image):
    global depth_frame_counter

    # --- 1. Logika .npy (Data Mentah) ---
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = np.reshape(array, (image.height, image.width, 4))
    R = array[:, :, 2].astype(np.float64)
    G = array[:, :, 1].astype(np.float64)
    B = array[:, :, 0].astype(np.float64)
    normalized = (R + G * 256.0 + B * 256.0 * 256.0) / (256.0 * 256.0 * 256.0 - 1.0)
    depth_in_meters = normalized * 1000.0
    np.save(f'D:/KP_Alexis_Doci/depth_raw/frame_{depth_frame_counter:05d}.npy', depth_in_meters)

    # --- 2. Logika .png (Gambar Visual/Original) ---
    image.convert(carla.ColorConverter.LogarithmicDepth)
    image.save_to_disk(f'D:/KP_Alexis_Doci/depth/frame_{depth_frame_counter:05d}.png')

    # Pindahkan counter ke paling akhir
    depth_frame_counter += 1

def process_lidar(lidar_data):
    """Menyimpan data mentah bird's-eye Lidar (X,Y,Z,I) ke .npy"""
    global lidar_frame_counter

    # Dapatkan raw data (buffer) dan ubah ke numpy array float32
    points = np.frombuffer(lidar_data.raw_data, dtype=np.dtype('f4'))

    # Reshape menjadi (N, 4) -> N adalah jumlah titik, 4 adalah (X, Y, Z, Intensitas)
    points = np.reshape(points, (-1, 4))

    # Simpan sebagai file .npy
    np.save(f'D:/KP_Alexis_Doci/lidar_raw/frame_{lidar_frame_counter:05d}.npy', points)
    lidar_frame_counter += 1


# === Window Car view callbacks ===
def process_rgb_window(image):
    global rgb_frame_counter_window
    image.save_to_disk(f'D:/KP_Alexis_Doci/rgb_window/frame_{rgb_frame_counter_window:05d}.png')
    rgb_frame_counter_window += 1

def process_seg_window(image):
    global seg_frame_counter_window

    # 1. Konversi ke CityScapesPalette dan Simpan PNG Visual
    image.convert(carla.ColorConverter.CityScapesPalette)
    image.save_to_disk(f'D:/KP_Alexis_Doci/seg_window/frame_{seg_frame_counter_window:05d}.png')

    # 2. Ambil gambar yang baru disimpan dan Petakan ke Class ID
    visual_path = f'D:/KP_Alexis_Doci/seg_window/frame_{seg_frame_counter_window:05d}.png'
    visual_img = Image.open(visual_path).convert('RGB')
    visual_array = np.array(visual_img)
    mapped_array = np.zeros((image.height, image.width), dtype=np.uint8)

    # Lakukan Mapping
    for rgb_color, class_id in CITYSCAPES_LEGEND.items():
        mask = np.all(visual_array == rgb_color, axis=-1)
        mapped_array[mask] = class_id

    # 3. Simpan Class ID Array ke NPY
    np.save(f'D:/KP_Alexis_Doci/seg_window_raw/frame_{seg_frame_counter_window:05d}.npy', mapped_array)

    seg_frame_counter_window += 1

def process_depth_window(image):
    global depth_frame_counter_window

    # --- 1. Logika .npy (Data Mentah) ---
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = np.reshape(array, (image.height, image.width, 4))
    R = array[:, :, 2].astype(np.float64)
    G = array[:, :, 1].astype(np.float64)
    B = array[:, :, 0].astype(np.float64)
    normalized = (R + G * 256.0 + B * 256.0 * 256.0) / (256.0 * 256.0 * 256.0 - 1.0)
    depth_in_meters = normalized * 1000.0
    np.save(f'D:/KP_Alexis_Doci/depth_window_raw/frame_{depth_frame_counter_window:05d}.npy', depth_in_meters)

    # --- 2. Logika .png (Gambar Visual/Original) ---
    image.convert(carla.ColorConverter.LogarithmicDepth)
    image.save_to_disk(f'D:/KP_Alexis_Doci/depth_window/frame_{depth_frame_counter_window:05d}.png')

    # Pindahkan counter ke paling akhir
    depth_frame_counter_window += 1

    # Simpan ke folder _raw
    np.save(f'D:/KP_Alexis_Doci/depth_window_raw/frame_{depth_frame_counter_window:05d}.npy', depth_in_meters)

    # Baris lama dinonaktifkan:
    # image.convert(carla.ColorConverter.LogarithmicDepth)
    # image.save_to_disk(f'D:/KP_Alexis_Doci/depth_window/frame_{depth_frame_counter_window:05d}.png')

    depth_frame_counter_window += 1

def process_lidar_window(lidar_data):
    """Menyimpan data mentah forward-facing Lidar (X,Y,Z,I) ke .npy"""
    global lidar_frame_counter_window

    points = np.frombuffer(lidar_data.raw_data, dtype=np.dtype('f4'))
    points = np.reshape(points, (-1, 4))

    np.save(f'D:/KP_Alexis_Doci/lidar_window_raw/frame_{lidar_frame_counter_window:05d}.npy', points)
    lidar_frame_counter_window += 1

In [ ]:
def process_semantic_lidar_data(lidar_data):
    """Fungsi khusus untuk mengoleksi data LiDAR Semantik ke CSV"""
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")

    # Format Semantic LiDAR CARLA (24 byte per titik)
    points = np.frombuffer(lidar_data.raw_data, dtype=np.dtype([
        ('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
        ('cos', 'f4'), ('idx', 'u4'), ('tag', 'u4')
    ]))

    # 1. Simpan Raw Point Cloud ke CSV
    raw_points = [[ts, p['x'], p['y'], p['z'], p['tag']] for p in points]
    with open(RAW_LIDAR_CSV, 'a', newline='') as f:
        csv.writer(f).writerows(raw_points)

    # 2. Hitung jarak terdekat untuk 23 kategori objek
    nearest = {}
    for p in points:
        tag = p['tag']
        name = key_list[value_list.index(tag)] if tag in value_list else f"Unknown({tag})"
        dist = math.sqrt(p['x']**2 + p['y']**2 + p['z']**2)
        if name not in nearest or dist < nearest[name]:
            nearest[name] = dist

    # 3. Simpan log deteksi ke object_log.csv (isi -1 jika objek tidak ditemukan)
    row = [ts] + [round(nearest.get(k, -1), 3) for k in key_list]
    with open(DETECTION_LOG_CSV, 'a', newline='') as f:
        csv.writer(f).writerow(row)

In [ ]:
#send the car off on autopilot - this will leave the spectator
vehicle.set_autopilot(True)

In [ ]:
import csv
import time
from queue import Queue
from queue import Empty
from matplotlib import cm # Diperlukan untuk colormap proyeksi
from PIL import Image # Diperlukan untuk menyimpan gambar proyeksi

# Siapkan file CSV untuk logging (dari skrip Anda)
log_filename = f'car_location_log_{time.strftime("%Y%m%d-%H%M%S")}.csv'
log_file = open(log_filename, 'w', newline='')
log_writer = csv.writer(log_file)
log_writer.writerow(['frame', 'timestamp', 'x', 'y', 'z'])

# =================================================================
# === PENGATURAN UNTUK LOGIKA PROYEKSI (DARI 'def tutorial') ===
# =================================================================

# Buat Antrian (Queues) untuk data sensor yang sinkron
print("Menyiapkan antrian data sensor...")
image_queue_bev = Queue()    # Untuk gambar Bird's-Eye (camera_rgb)
image_queue_window = Queue() # BARU: Untuk gambar Window (camera_rgb_window)
lidar_queue_window = Queue() # Untuk Lidar depan (camera_lidar_window)

# Sensor yang digunakan:
# 1. camera_rgb (Bird's-eye cam)
# 2. camera_rgb_window (Window/forward cam)
# 3. camera_lidar_window (Window/forward Lidar)

# Arahkan sensor yang relevan untuk memasukkan data ke Queues
camera_rgb.listen(lambda data: (process_rgb(data), image_queue_bev.put(data)))
camera_lidar_window.listen(lambda data: (process_lidar_window(data), lidar_queue_window.put(data)))
# LiDAR Semantik: kirim data ke pemroses CSV
camera_lidar_sem.listen(lambda data: process_semantic_lidar_data(data))
# (Kita akan mengatur listener camera_rgb_window di bawah setelah listener lainnya)

# Tentukan path dasar Anda
base_path = "D:/KP_Alexis_Doci"

# Buat direktori output untuk gambar proyeksi
os.makedirs('_out', exist_ok=True)
os.makedirs('_out_window', exist_ok=True) # BARU: Folder output untuk proyeksi window

print("Membuat direktori output untuk data mentah...")
# (Folder-folder ini sudah ada di skrip Anda, pastikan path-nya benar)
os.makedirs(f'{base_path}/lidar_raw', exist_ok=True)
os.makedirs(f'{base_path}/lidar_window_raw', exist_ok=True)
os.makedirs(f'{base_path}/seg_raw', exist_ok=True)
os.makedirs(f'{base_path}/depth_raw', exist_ok=True)
os.makedirs(f'{base_path}/rgb_window_raw', exist_ok=True)
os.makedirs(f'{base_path}/seg_window_raw', exist_ok=True)
os.makedirs(f'{base_path}/depth_window_raw', exist_ok=True)


# --- Pengaturan Proyeksi 1: Bird's-Eye (BEV) ---
print("Menghitung matriks intrinsik K untuk kamera BEV...")
VIRIDIS = np.array(cm.get_cmap('viridis').colors)
VID_RANGE = np.linspace(0.0, 1.0, VIRIDIS.shape[0])

image_w_bev = int(camera_rgb.attributes['image_size_x'])
image_h_bev = int(camera_rgb.attributes['image_size_y'])
fov_bev = float(camera_rgb.attributes['fov'])
focal_bev = image_w_bev / (2.0 * np.tan(fov_bev * np.pi / 360.0))

K_bev = np.identity(3)
K_bev[0, 0] = K_bev[1, 1] = focal_bev
K_bev[0, 2] = image_w_bev / 2.0
K_bev[1, 2] = image_h_bev / 2.0

# --- Pengaturan Proyeksi 2: Window ---
print("Menghitung matriks intrinsik K untuk kamera Window...")
image_w_window = int(camera_rgb_window.attributes['image_size_x'])
image_h_window = int(camera_rgb_window.attributes['image_size_y'])
fov_window = float(camera_rgb_window.attributes['fov'])
focal_window = image_w_window / (2.0 * np.tan(fov_window * np.pi / 360.0))

K_window = np.identity(3)
K_window[0, 0] = K_window[1, 1] = focal_window
K_window[0, 2] = image_w_window / 2.0
K_window[1, 2] = image_h_window / 2.0


# Ukuran titik untuk visualisasi
dot_extent = 1 # (default 2-1 = 1)

# =================================================================
# === AKHIR DARI PENGATURAN PROYEKSI ===
# =================================================================

# Mulai listener untuk 5 sensor lainnya (untuk penyimpanan data biasa)
print("Memulai listener sensor lainnya...")
camera_seg.listen(process_seg)
camera_depth.listen(process_depth)
camera_lidar.listen(process_lidar) # Lidar bird's-eye

# --- Listener yang Dimodifikasi ---
# camera_rgb_window sekarang melakukan DUA hal:
# 1. Menjalankan process_rgb_window (menyimpan PNG biasa)
# 2. Memasukkan data ke image_queue_window (untuk proyeksi)
camera_rgb_window.listen(lambda data: (process_rgb_window(data), image_queue_window.put(data)))

camera_seg_window.listen(process_seg_window)
camera_depth_window.listen(process_depth_window)

# Mulai perekam (dari skrip Anda)
client.start_recorder("D:/KP_Alexis_Doci/recording01.log")

print("\nSimulasi, logging, dan PROYEKSI GANDA Lidar dimulai...")
print("Tekan Ctrl+C (atau tombol Stop di Jupyter) untuk berhenti.")

try:
    while True:
        # Dapatkan snapshot SEBELUM tick (untuk logging data)
        snapshot = world.get_snapshot()

        # Tick dunia (ini adalah detak jantung simulasi)
        world.tick()

        # LOGIKA 1: Logging Lokasi
        if vehicle is not None and vehicle.is_alive:
            loc = vehicle.get_transform().location
            log_writer.writerow([snapshot.frame, snapshot.timestamp.elapsed_seconds, loc.x, loc.y, loc.z])

        # --- LOGIKA 2: Proyeksi Lidar Ganda ---
        try:
            # Dapatkan data sinkron dari antrian
            image_data_bev = image_queue_bev.get(True, 0.05)     # Gambar BEV
            image_data_window = image_queue_window.get(True, 0.05) # Gambar Window
            lidar_data = lidar_queue_window.get(True, 0.05)        # Data Lidar

            # --- Pemrosesan LiDAR (Dilakukan SEKALI) ---
            # (Data ini akan digunakan untuk kedua proyeksi)
#             p_cloud = np.frombuffer(lidar_data_std.raw_data, dtype=np.dtype('f4')).reshape(-1, 4)
            p_cloud = np.copy(np.frombuffer(lidar_data.raw_data, dtype=np.dtype('f4')))
            p_cloud = np.reshape(p_cloud, (-1, 4))
            intensity = p_cloud[:, 3]
            local_lidar_points = np.array(p_cloud[:, :3]).T
            local_lidar_points = np.r_[
                local_lidar_points, [np.ones(local_lidar_points.shape[1])]]

            # (1) Lidar (camera_lidar_window) ke World
            lidar_2_world = camera_lidar_window.get_transform().get_matrix()
            world_points = np.dot(lidar_2_world, local_lidar_points)

            # =======================================================
            # === PROYEKSI 1: Ke Kamera Bird's-Eye (BEV) ===
            # =======================================================

            # --- Proses Gambar BEV ---
            im_array_bev = np.copy(np.frombuffer(image_data_bev.raw_data, dtype=np.dtype("uint8")))
            im_array_bev = np.reshape(im_array_bev, (image_data_bev.height, image_data_bev.width, 4))
            im_array_bev = im_array_bev[:, :, :3][:, :, ::-1] # BGRA -> RGB

            # --- Proyeksi Lanjutan (BEV) ---
            # (2) World ke Kamera BEV (camera_rgb)
            world_2_camera_bev = np.array(camera_rgb.get_transform().get_inverse_matrix())
            sensor_points_bev = np.dot(world_2_camera_bev, world_points)

            # (3) Ubah koordinat UE4 ke standar
            point_in_camera_coords_bev = np.array([
                sensor_points_bev[1],
                sensor_points_bev[2] * -1,
                sensor_points_bev[0]])

            # (4) Proyeksi 3D -> 2D menggunakan K_bev
            points_2d_bev = np.dot(K_bev, point_in_camera_coords_bev)
            points_2d_bev = np.array([
                points_2d_bev[0, :] / points_2d_bev[2, :],
                points_2d_bev[1, :] / points_2d_bev[2, :],
                points_2d_bev[2, :]])

            # --- Filter dan Visualisasikan Titik (BEV) ---
            points_2d_bev = points_2d_bev.T
            intensity_bev = intensity.T # Salin data intensitas

            points_in_canvas_mask_bev = \
                (points_2d_bev[:, 0] > 0.0) & (points_2d_bev[:, 0] < image_w_bev) & \
                (points_2d_bev[:, 1] > 0.0) & (points_2d_bev[:, 1] < image_h_bev) & \
                (points_2d_bev[:, 2] > 0.0)

            points_2d_bev = points_2d_bev[points_in_canvas_mask_bev]
            intensity_bev = intensity_bev[points_in_canvas_mask_bev]

            if points_2d_bev.shape[0] > 0:
                u_coord_bev = points_2d_bev[:, 0].astype(int)
                v_coord_bev = points_2d_bev[:, 1].astype(int)

                # Warnai titik
                intensity_bev = 4 * intensity_bev - 3
                color_map_bev = np.array([
                    np.interp(intensity_bev, VID_RANGE, VIRIDIS[:, 0]) * 255.0,
                    np.interp(intensity_bev, VID_RANGE, VIRIDIS[:, 1]) * 255.0,
                    np.interp(intensity_bev, VID_RANGE, VIRIDIS[:, 2]) * 255.0]).astype(int).T

                # Gambar titik di gambar
                if dot_extent <= 0:
                    im_array_bev[v_coord_bev, u_coord_bev] = color_map_bev
                else:
                    for i in range(len(points_2d_bev)):
                        im_array_bev[
                            v_coord_bev[i]-dot_extent : v_coord_bev[i]+dot_extent,
                            u_coord_bev[i]-dot_extent : u_coord_bev[i]+dot_extent] = color_map_bev[i]

            # --- Simpan gambar proyeksi BEV ---
            image_bev = Image.fromarray(im_array_bev)
            image_bev.save("D:/KP_Alexis_Doci/lidar_frames/proj_%08d.png" % image_data_bev.frame)


            # =======================================================
            # === PROYEKSI 2: Ke Kamera Window (Depan) ===
            # =======================================================

            # --- Proses Gambar Window ---
            im_array_window = np.copy(np.frombuffer(image_data_window.raw_data, dtype=np.dtype("uint8")))
            im_array_window = np.reshape(im_array_window, (image_data_window.height, image_data_window.width, 4))
            im_array_window = im_array_window[:, :, :3][:, :, ::-1] # BGRA -> RGB

            # --- Proyeksi Lanjutan (Window) ---
            # (2) World ke Kamera Window (camera_rgb_window)
            world_2_camera_window = np.array(camera_rgb_window.get_transform().get_inverse_matrix())
            sensor_points_window = np.dot(world_2_camera_window, world_points)

            # (3) Ubah koordinat UE4 ke standar
            point_in_camera_coords_window = np.array([
                sensor_points_window[1],
                sensor_points_window[2] * -1,
                sensor_points_window[0]])

            # (4) Proyeksi 3D -> 2D menggunakan K_window
            points_2d_window = np.dot(K_window, point_in_camera_coords_window)
            points_2d_window = np.array([
                points_2d_window[0, :] / points_2d_window[2, :],
                points_2d_window[1, :] / points_2d_window[2, :],
                points_2d_window[2, :]])

            # --- Filter dan Visualisasikan Titik (Window) ---
            points_2d_window = points_2d_window.T
            intensity_window = intensity.T # Salin data intensitas

            points_in_canvas_mask_window = \
                (points_2d_window[:, 0] > 0.0) & (points_2d_window[:, 0] < image_w_window) & \
                (points_2d_window[:, 1] > 0.0) & (points_2d_window[:, 1] < image_h_window) & \
                (points_2d_window[:, 2] > 0.0)

            points_2d_window = points_2d_window[points_in_canvas_mask_window]
            intensity_window = intensity_window[points_in_canvas_mask_window]

            if points_2d_window.shape[0] > 0:
                u_coord_window = points_2d_window[:, 0].astype(int)
                v_coord_window = points_2d_window[:, 1].astype(int)

                # Warnai titik
                intensity_window = 4 * intensity_window - 3
                color_map_window = np.array([
                    np.interp(intensity_window, VID_RANGE, VIRIDIS[:, 0]) * 255.0,
                    np.interp(intensity_window, VID_RANGE, VIRIDIS[:, 1]) * 255.0,
                    np.interp(intensity_window, VID_RANGE, VIRIDIS[:, 2]) * 255.0]).astype(int).T

                # Gambar titik di gambar
                if dot_extent <= 0:
                    im_array_window[v_coord_window, u_coord_window] = color_map_window
                else:
                    for i in range(len(points_2d_window)):
                        im_array_window[
                            v_coord_window[i]-dot_extent : v_coord_window[i]+dot_extent,
                            u_coord_window[i]-dot_extent : u_coord_window[i]+dot_extent] = color_map_window[i]

            # --- Simpan gambar proyeksi Window ---
            image_window = Image.fromarray(im_array_window)
            image_window.save("D:/KP_Alexis_Doci/lidar_window/proj_%08d.png" % image_data_window.frame)


        except Empty:
            # Ini akan terjadi jika salah satu antrian kosong (frame terlewat)
            # Kita 'continue' agar loop utama tetap berjalan
            continue

except KeyboardInterrupt:
    print("\nInterupsi oleh pengguna. Membersihkan...")
finally:
    # Hentikan semua sensor
    print("Menghentikan semua sensor...")
    camera_rgb.stop()
    camera_seg.stop()
    camera_depth.stop()
    camera_lidar.stop()
    camera_rgb_window.stop()
    camera_seg_window.stop()
    camera_depth_window.stop()
    camera_lidar_window.stop()


    client.stop_recorder()
    log_file.close()
    print("Semua sensor dihentikan dan file log ditutup.")

    # --- Tambahan: Simpan Info Simulasi ke TXT ---
    try:
        with open(INFO_LOG_TXT, "w") as f:
            f.write("=== Simulation Info (Integrated Notebook) ===\n")
            f.write(f"Start: {sim_start_time}\n")
            f.write(f"End:   {datetime.now()}\n")
            f.write(f"Map:   {world.get_map().name}\n")
            f.write(f"Vehicle model: {vehicle.type_id}\n")
            f.write("\n--- Camera/Spectator Transform ---\n")
            f.write(str(world.get_spectator().get_transform()) + "\n")
        print(f"Info simulasi berhasil disimpan ke {INFO_LOG_TXT}")
    except Exception as e:
        print(f"Gagal menyimpan info simulasi: {e}")

    camera_lidar_sem.stop()

    # Kembalikan pengaturan dunia
    if synchronous_master:
        print("Mengembalikan pengaturan dunia...")
        settings = world.get_settings()
        settings.synchronous_mode = False
        settings.fixed_delta_seconds = None
        world.apply_settings(settings)

    # Hancurkan semua aktor yang telah di-spawn
    print("Menghancurkan semua aktor yang di-spawn...")

    actor_ids_to_destroy = []
    actor_ids_to_destroy.extend(vehicles_list) # 30 mobil
    for w in walkers_list: # 10 pejalan kaki + 10 controller
        actor_ids_to_destroy.append(w["id"])
        actor_ids_to_destroy.append(w["con"])

    # Tambahkan mobil ego dan sensor-sensornya
    if 'vehicle' in locals() and vehicle is not None and vehicle.is_alive:
         actor_ids_to_destroy.append(vehicle.id) # Mobil Audi
         # Sensor akan hancur bersama mobil 'ego' karena 'attach_to'

    # Buat batch command untuk menghancurkan (Filter ID yang mungkin sudah hancur)
    batch_destroy = [carla.command.DestroyActor(actor_id) for actor_id in actor_ids_to_destroy if world.get_actor(actor_id) is not None]

    if batch_destroy:
        print(f"Menghancurkan {len(batch_destroy)} aktor...")
        try:
            client.apply_batch(batch_destroy, True) # Hancurkan secara sinkron
        except Exception as e:
            print(f"Error saat menghancurkan aktor: {e}")

    print("Selesai.")

Menyiapkan antrian data sensor...
Membuat direktori output untuk data mentah...
Menghitung matriks intrinsik K untuk kamera BEV...
Menghitung matriks intrinsik K untuk kamera Window...
Memulai listener sensor lainnya...

Simulasi, logging, dan PROYEKSI GANDA Lidar dimulai...
Tekan Ctrl+C (atau tombol Stop di Jupyter) untuk berhenti.

Interupsi oleh pengguna. Membersihkan...
Menghentikan semua sensor...
Semua sensor dihentikan dan file log ditutup.
Info simulasi berhasil disimpan ke D:/KP_Alexis_Doci/simulation_info.txt
Mengembalikan pengaturan dunia...
Menghancurkan semua aktor yang di-spawn...
Menghancurkan 47 aktor...
Selesai.


In [ ]:
# HANYA STORE KODE LAMA, TIDAK DIPAKAI
# import csv
# import time
# from queue import Queue
# from queue import Empty
# from matplotlib import cm # Diperlukan untuk colormap proyeksi
# from PIL import Image # Diperlukan untuk menyimpan gambar proyeksi

# # Siapkan file CSV untuk logging (dari skrip Anda)
# log_filename = f'car_location_log_{time.strftime("%Y%m%d-%H%M%S")}.csv'
# log_file = open(log_filename, 'w', newline='')
# log_writer = csv.writer(log_file)
# log_writer.writerow(['frame', 'timestamp', 'x', 'y', 'z'])

# # =================================================================
# # === PENGATURAN UNTUK LOGIKA PROYEKSI (DARI 'def tutorial') ===
# # =================================================================

# # Buat Antrian (Queues) untuk data sensor yang sinkron
# # Kita akan memproyeksikan data Lidar *window* (menghadap ke depan)
# # ke data kamera *bird's-eye* (menghadap ke bawah).
# image_queue = Queue()
# lidar_queue = Queue()

# # Sensor yang digunakan (diasumsikan sudah di-spawn di sel sebelumnya):
# # 1. camera_rgb: Bird's-eye camera (dari Sel 38)
# # 2. camera_lidar_window: Window/forward-facing Lidar (dari Sel 39)

# # Arahkan sensor yang relevan untuk memasukkan data ke Queues
# camera_rgb.listen(lambda data: (process_rgb(data), image_queue.put(data)))
# camera_lidar_window.listen(lambda data: (process_lidar_window(data), lidar_queue.put(data)))
# # Tentukan path dasar Anda
# base_path = "D:/KP_Alexis_Doci"

# # Buat direktori output untuk gambar proyeksi
# os.makedirs('_out', exist_ok=True)

# print("Membuat direktori output untuk data mentah...")

# # Folder dari traceback error Anda
# os.makedirs(f'{base_path}/lidar_raw', exist_ok=True)
# os.makedirs(f'{base_path}/lidar_window_raw', exist_ok=True)

# # Berdasarkan fungsi 'process_...' lain yang Anda panggil,
# # Anda kemungkinan besar juga memerlukan folder-folder ini:
# os.makedirs(f'{base_path}/seg_raw', exist_ok=True)
# os.makedirs(f'{base_path}/depth_raw', exist_ok=True)
# os.makedirs(f'{base_path}/rgb_window_raw', exist_ok=True)
# os.makedirs(f'{base_path}/seg_window_raw', exist_ok=True)
# os.makedirs(f'{base_path}/depth_window_raw', exist_ok=True)

# # Dapatkan konstanta colormap untuk Lidar
# VIRIDIS = np.array(cm.get_cmap('viridis').colors)
# VID_RANGE = np.linspace(0.0, 1.0, VIRIDIS.shape[0])

# # Dapatkan parameter kamera (dari camera_rgb bird's-eye) untuk matriks K
# image_w = int(camera_rgb.attributes['image_size_x'])
# image_h = int(camera_rgb.attributes['image_size_y'])
# fov = float(camera_rgb.attributes['fov'])
# focal = image_w / (2.0 * np.tan(fov * np.pi / 360.0))

# # Bangun matriks K
# K = np.identity(3)
# K[0, 0] = K[1, 1] = focal
# K[0, 2] = image_w / 2.0
# K[1, 2] = image_h / 2.0

# # Ukuran titik untuk visualisasi
# dot_extent = 1 # (default 2-1 = 1)

# # =================================================================
# # === AKHIR DARI PENGATURAN PROYEKSI ===
# # =================================================================

# # Mulai listener untuk 6 sensor lainnya (untuk penyimpanan data biasa)
# camera_seg.listen(process_seg)
# camera_depth.listen(process_depth)
# camera_lidar.listen(process_lidar) # Lidar bird's-eye
# camera_rgb_window.listen(process_rgb_window) # Kamera depan
# camera_seg_window.listen(process_seg_window)
# camera_depth_window.listen(process_depth_window)

# # Mulai perekam (dari skrip Anda)
# client.start_recorder("D:/KP_Alexis_Doci/recording01.log")

# print("Simulasi, logging, dan proyeksi Lidar dimulai...")
# print("Tekan Ctrl+C (atau tombol Stop di Jupyter) untuk berhenti.")

# try:
#     while True:
#         # Dapatkan snapshot SEBELUM tick (untuk logging data)
#         snapshot = world.get_snapshot()

#         # Tick dunia (ini adalah detak jantung simulasi)
#         world.tick()

#         # --- LOGIKA 1: Logging CSV (Diperbaiki) ---
#         # (Kode asli Anda tidak memanggil fungsi logging, ini memperbaikinya)
#         if vehicle is not None and vehicle.is_alive:
#             loc = vehicle.get_transform().location
#             timestamp = snapshot.timestamp
#             log_writer.writerow([
#                 snapshot.frame,
#                 timestamp.elapsed_seconds,
#                 loc.x, loc.y, loc.z
#             ])

#         # --- LOGIKA 2: Proyeksi Lidar (Baru) ---
#         try:
#             # Dapatkan data sinkron dari antrian
#             image_data = image_queue.get(True, 0.05) # timeout 50ms
#             lidar_data = lidar_queue.get(True, 0.05)

#             # --- Pemrosesan Gambar ---
#             im_array = np.copy(np.frombuffer(image_data.raw_data, dtype=np.dtype("uint8")))
#             im_array = np.reshape(im_array, (image_data.height, image_data.width, 4))
#             im_array = im_array[:, :, :3][:, :, ::-1] # BGRA -> RGB

#             # --- Pemrosesan LiDAR ---
#             p_cloud = np.copy(np.frombuffer(lidar_data.raw_data, dtype=np.dtype('f4')))
#             p_cloud = np.reshape(p_cloud, (-1, 4))
#             intensity = p_cloud[:, 3]
#             local_lidar_points = np.array(p_cloud[:, :3]).T
#             local_lidar_points = np.r_[
#                 local_lidar_points, [np.ones(local_lidar_points.shape[1])]]

#             # --- Proyeksi LiDAR ke Kamera ---

#             # (1) Lidar (camera_lidar_window) ke World
#             lidar_2_world = camera_lidar_window.get_transform().get_matrix()
#             world_points = np.dot(lidar_2_world, local_lidar_points)

#             # (2) World ke Kamera (camera_rgb)
#             world_2_camera = np.array(camera_rgb.get_transform().get_inverse_matrix())
#             sensor_points = np.dot(world_2_camera, world_points)

#             # (3) Ubah koordinat UE4 ke standar
#             point_in_camera_coords = np.array([
#                 sensor_points[1],
#                 sensor_points[2] * -1,
#                 sensor_points[0]])

#             # (4) Proyeksi 3D -> 2D menggunakan K
#             points_2d = np.dot(K, point_in_camera_coords)
#             points_2d = np.array([
#                 points_2d[0, :] / points_2d[2, :],
#                 points_2d[1, :] / points_2d[2, :],
#                 points_2d[2, :]])

#             # --- Filter dan Visualisasikan Titik ---
#             points_2d = points_2d.T
#             intensity = intensity.T

#             points_in_canvas_mask = \
#                 (points_2d[:, 0] > 0.0) & (points_2d[:, 0] < image_w) & \
#                 (points_2d[:, 1] > 0.0) & (points_2d[:, 1] < image_h) & \
#                 (points_2d[:, 2] > 0.0)

#             points_2d = points_2d[points_in_canvas_mask]
#             intensity = intensity[points_in_canvas_mask]

#             u_coord = points_2d[:, 0].astype(int)
#             v_coord = points_2d[:, 1].astype(int)

#             # --- Warnai titik ---
#             intensity = 4 * intensity - 3
#             color_map = np.array([
#                 np.interp(intensity, VID_RANGE, VIRIDIS[:, 0]) * 255.0,
#                 np.interp(intensity, VID_RANGE, VIRIDIS[:, 1]) * 255.0,
#                 np.interp(intensity, VID_RANGE, VIRIDIS[:, 2]) * 255.0]).astype(int).T

#             # --- Gambar titik di gambar ---
#             if dot_extent <= 0:
#                 im_array[v_coord, u_coord] = color_map
#             else:
#                 for i in range(len(points_2d)):
#                     im_array[
#                         v_coord[i]-dot_extent : v_coord[i]+dot_extent,
#                         u_coord[i]-dot_extent : u_coord[i]+dot_extent] = color_map[i]

#             # --- Simpan gambar proyeksi ---
#             image = Image.fromarray(im_array)
#             image.save("_out/proj_%08d.png" % image_data.frame)

#         except Empty:
#             # Ini akan terjadi jika antrian kosong (frame terlewat)
#             # Kita 'continue' agar loop utama tetap berjalan
#             continue

# except KeyboardInterrupt:
#     print("\nInterupsi oleh pengguna. Membersihkan...")
# finally:
#     # Hentikan semua sensor
#     print("Menghentikan semua sensor...")
#     camera_rgb.stop()
#     camera_seg.stop()
#     camera_depth.stop()
#     camera_lidar.stop()
#     camera_rgb_window.stop()
#     camera_seg_window.stop()
#     camera_depth_window.stop()
#     camera_lidar_window.stop()

#     client.stop_recorder()
#     log_file.close()
#     print("Semua sensor dihentikan dan file log ditutup.")

#     # Kembalikan pengaturan dunia
#     if synchronous_master:
#         print("Mengembalikan pengaturan dunia...")
#         settings = world.get_settings()
#         settings.synchronous_mode = False
#         settings.fixed_delta_seconds = None
#         world.apply_settings(settings)

#     # Hancurkan semua aktor yang telah di-spawn
#     # (Skrip Anda tidak melakukan ini, ini penting!)
#     print("Menghancurkan semua aktor yang di-spawn...")

#     # Kumpulkan ID semua aktor
#     actor_ids_to_destroy = []
#     actor_ids_to_destroy.extend(vehicles_list) # 30 mobil
#     for w in walkers_list: # 10 pejalan kaki + 10 controller
#         actor_ids_to_destroy.append(w["id"])
#         actor_ids_to_destroy.append(w["con"])

#     # Tambahkan mobil ego dan sensor-sensornya
#     if vehicle is not None:
#          actor_ids_to_destroy.append(vehicle.id) # Mobil Audi
#          # Sensor akan hancur bersama mobil 'ego' karena 'attach_to'

#     # Buat batch command untuk menghancurkan
#     batch_destroy = [carla.command.DestroyActor(actor_id) for actor_id in actor_ids_to_destroy if world.get_actor(actor_id) is not None]

#     if batch_destroy:
#         client.apply_batch(batch_destroy, True) # Hancurkan secara sinkron

#     print(f"Selesai. {len(batch_destroy)} aktor dihancurkan.")

In [ ]:
# client.replay_file("D:/KP_Alexis_Doci/recording01.log", 0, 0, 0)